In [ ]:
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import io
import os

# --- CONFIGURATION ---
# Change this to your compiled executable name (e.g., "benchmark_app.exe" for Windows)
EXECUTABLE_PATH = "./benchmark_app" 

# Default values
DEFAULT_N = 500000
DEFAULT_PROB = 0.0001 # Changed from 0.4 to prevent RAM crash, adjust as needed!
DEFAULT_THREADS = 8

def run_cpp_benchmark(n, prob, p):
    """Writes params to file, runs C++, and returns a DataFrame of the results."""
    # Write parameters for C++ to read
    with open("params.txt", "w") as f:
        f.write(f"{int(n)} {float(prob)} {int(p)}\n")
    
    # Execute C++ program
    print(f"Running: N={n}, Prob={prob}, Threads={p}...")
    result = subprocess.run([EXECUTABLE_PATH], capture_output=True, text=True)
    
    if result.returncode != 0:
        print("Error running C++ code:\n", result.stderr)
        return pd.DataFrame()
    
    # Read the CSV output directly into Pandas
    try:
        df = pd.read_csv(io.StringIO(result.stdout.strip()))
        df['N'] = n
        df['Prob'] = prob
        df['Threads'] = p
        return df
    except Exception as e:
        print("Failed to parse output:", result.stdout)
        return pd.DataFrame()

# --- SWEEP DEFINITIONS ---
thread_sweep = [2, 4, 8, 16, 32]
vertex_sweep = [100000, 200000, 300000, 400000, 500000]
prob_sweep   = [0.2, 0.4, 0.6, 0.8] # Warning: Do not run this on N=500,000 unless you have a supercomputer!

# --- EXECUTE SWEEPS ---
results_threads = []
print("--- STARTING THREAD SWEEP ---")
for t in thread_sweep:
    results_threads.append(run_cpp_benchmark(DEFAULT_N, DEFAULT_PROB, t))
df_threads = pd.concat(results_threads, ignore_index=True)

results_vertices = []
print("\n--- STARTING VERTEX SWEEP ---")
for v in vertex_sweep:
    results_vertices.append(run_cpp_benchmark(v, DEFAULT_PROB, DEFAULT_THREADS))
df_vertices = pd.concat(results_vertices, ignore_index=True)

results_prob = []
print("\n--- STARTING PROBABILITY SWEEP ---")
# Overriding DEFAULT_N for probability sweep to prevent system crash
SAFE_DENSE_N = 5000 
for p_val in prob_sweep:
    results_prob.append(run_cpp_benchmark(SAFE_DENSE_N, p_val, DEFAULT_THREADS))
df_prob = pd.concat(results_prob, ignore_index=True)


# --- PLOTTING DATA ---
def plot_benchmark(df, x_col, x_label, title, log_y=True):
    plt.figure(figsize=(12, 7))
    
    algorithms = df['Algorithm'].unique()
    for algo in algorithms:
        subset = df[df['Algorithm'] == algo]
        plt.plot(subset[x_col], subset['Time_us'] / 1000, marker='o', label=algo, linewidth=2)
        
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel("Execution Time (Milliseconds)", fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    
    if log_y:
        plt.yscale('log')
        plt.ylabel("Execution Time (Milliseconds) - LOG SCALE", fontsize=12)
        
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

# 1. Plot Time vs Threads
plot_benchmark(df_threads, 'Threads', 'Number of Threads', 
               f'Scalability: Time vs. Threads\n(N={DEFAULT_N}, Prob={DEFAULT_PROB})', log_y=True)

# 2. Plot Time vs Vertices
plot_benchmark(df_vertices, 'N', 'Number of Vertices (N)', 
               f'Time vs. Graph Size\n(Threads={DEFAULT_THREADS}, Prob={DEFAULT_PROB})', log_y=False)

# 3. Plot Time vs Probability
plot_benchmark(df_prob, 'Prob', 'Edge Probability', 
               f'Time vs. Graph Density\n(N={SAFE_DENSE_N}, Threads={DEFAULT_THREADS})', log_y=False)

# Optional: Print raw data tables
print("\nFinal Data: Thread Sweep")
display(df_threads.pivot(index='Threads', columns='Algorithm', values='Time_us'))